In [15]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
import os

In [46]:
URL = "https://sandbox.oxylabs.io/products"
session = requests.Session()
headers = {
    "User-Agent": "Mozilla/5.0"
}
session.headers.update(headers)

In [17]:
# part 7
def request_catering(url, params=None):
    for i in range(5):
        try:
            if params:
                response = session.get(url, params=params, timeout=10, allow_redirects=True)
            else:
                response = session.get(url, timeout=10, allow_redirects=True)

            if response.status_code == 200:
                return response
            else:
                print("Request failed:", response.status_code,"Attempt:",i + 1, "for", url)
        except requests.RequestException as e:
            print("Error:", e,"Attempt:",i + 1)
            
        time.sleep(2)
    print("All retry attempts failed for:", url)
    return None

Reasoning:
 to make it resilient, we do the retry strategy and do it for a certain time of attempts to get if it gets OK response or not and if it still fails then the it jumps outside without crashing, and previous scraped products remain in the  "products" list

In [29]:

test_session = requests.Session()
test_session.headers.update(headers)

test_session.get(URL)  # establish session on page 1
session_response = test_session.get(URL, params={"page": 2})
fresh_response=requests.get(f"{URL}?page=2", headers=headers)
print("session response ->",session_response.url)
print("fresh response ->",fresh_response.url)
    

session response -> https://sandbox.oxylabs.io/products?page=2
fresh response -> https://sandbox.oxylabs.io/products?page=2


reasoning:


In [19]:
def productLinks(product_links):
    page_products=[]
    for link in product_links:

        href = link.get("href")
        if not href:
            continue

        if href.startswith("/products/") and not href.startswith("/products/category"):
            product_url = requests.compat.urljoin(current_url, href)

            # handling duplicates
            if product_url not in visited_products:
                visited_products.add(product_url)
                page_products.append(product_url)

    return page_products

In [20]:
def detail_page(page_products):

    for product_url in page_products:

        product_response = request_catering(product_url)

        if product_response is None:
            print("Could not load:", product_url)
            continue

        product_soup = BeautifulSoup(
            product_response.content,
            "html.parser"
        )

        # Product name
        name_tag = product_soup.find("h2", class_="title")

        if name_tag:
            name = name_tag.get_text(strip=True)
        else:
            name = ""


        # Price
        price_tag = product_soup.find(
            "div",
            class_="price"
        )

        if price_tag:
            price = price_tag.get_text(strip=True)
        else:
            price = ""


        # Stock status
        stock_tag = product_soup.find(
            "p",
            class_="availability"
        )

        if stock_tag:
            stock_status = stock_tag.get_text(strip=True).lower()
        else:
            stock_status = ""


        # Description
        description_tag = product_soup.find(
            "p",
            class_="description"
        )

        if description_tag:
            description = description_tag.get_text(
                " ",
                strip=True
            )
        else:
            description = ""


        products.append({
            "Product name": name,
            "Price": price,
            "Stock status": stock_status,
            "Description": description,
            "Detail-page URL": product_url,
            "Listing page number": page_number
        })

        print("  ", name)

In [ ]:
def csvchecking(filename):

    products = []
    visited_products = set()
    page_number = 1

    if os.path.exists(filename):

        print("found file!")
        old_df = pd.read_csv(filename)
        products = old_df.to_dict("records")

        if "Detail-page URL" in old_df.columns:
            visited_products = set(old_df["Detail-page URL"].dropna().astype(str))

        print("number of Products already saved:", len(products))

        # Find the last saved product
        if len(old_df) > 0:

            last_product_url = old_df["Detail-page URL"].iloc[-1]
            print("Last saved product:",last_product_url)

            #product number from url
            last_product_id = int(last_product_url.rstrip("/").split("/")[-1])
            # next product url
            next_product_url = (URL + "/" + str(last_product_id + 1))

            print("Next product to check:",next_product_url)


            # last page num
            page_number = int(old_df["Listing page number"].iloc[-1])
            print("Checking listing pages starting from:",page_number)


            # Find the listing page containing
            # the next product
            while True:

                if page_number == 1:
                    check_url = URL

                else:
                    check_url = (URL+ "?page="+ str(page_number))

                print("Checking listing page:",page_number)
                response = request_catering(check_url)

                if response is None:
                    print("Could not check page:",page_number)
                    return (products,visited_products,check_url,page_number)


                check_soup = BeautifulSoup(response.content,"html.parser")

                # all links
                links = check_soup.find_all("a")
                found_next_product = False

                for link in links:

                    href = link.get("href")

                    if not href:
                        continue

                    product_url = requests.compat.urljoin(check_url,href)

                    if product_url == next_product_url:
                        found_next_product = True
                        break


                if found_next_product:

                    print("Next product found on listing page:",page_number)
                    current_url = check_url
                    break

                page_number += 1


    else:
        print("No previous CSV found. Starting from beginning.")
        current_url = URL


    return (products,visited_products,current_url,page_number)

In [22]:
def save_progress(products,filename):
    # Save progress after every listing page
    df = pd.DataFrame(products)

    df = df.drop_duplicates(
        subset=["Detail-page URL"]
    )

    df.to_csv(filename, index=False)

    print("Progress saved. Total products:", len(df))

In [47]:
products = []

page_number = 1
visited_pages = set()
visited_products = set()
filename="23L_0570_versionA_static_products.csv"


products, visited_products, current_url, page_number = csvchecking(filename)

while current_url:

    
    if current_url in visited_pages:
        break

    visited_pages.add(current_url)

    print("Scraping listing page:", page_number)

    response = request_catering(current_url)

    if response is None:
        print("Could not load page. Stopping.")
        break

    soup = BeautifulSoup(response.content, "html.parser")

    # Find all product links
    product_links = soup.find_all("a")

    page_products = []

    page_products=productLinks(product_links)
    

    print("Products found:", len(page_products))

    detail_page(page_products)
    
    save_progress(products,filename)
    
    # Find Forward button
    forward_link = None

    for link in soup.find_all("a"):

        text = link.get_text(" ", strip=True).lower()

        if text == "forward":
            forward_link = link
            break


    
    if forward_link is None:
        print("Forward button not found.")
        break


    # Check if Forward button is disabled
    if forward_link.get("aria-disabled") == "true":
        print("Forward button is disabled.")
        print("Final page reached:", page_number)
        break


    href = forward_link.get("href")

    next_link = requests.compat.urljoin(
        current_url,
        href
    )


    current_url = next_link
    page_number += 1

    time.sleep(0.5)

Previous CSV found. Loading saved data...
Products already saved: 376
Last saved product: https://sandbox.oxylabs.io/products/420
Next product to check: https://sandbox.oxylabs.io/products/421
Checking listing pages starting from: 14
Checking listing page: 14
Next product found on listing page: 14
Scraping listing page: 14
Products found: 28
   Griftlands
   ZEN Pinball 2: Aliens Vs. Pinball
   Bit.Trip Complete
   Classic NES Series: Super Mario Bros.
   Age of Mythology: The Titans
   SimCity 4
   Sega Soccer Slam
   Spider-Man: Mysterio's Menace
   Super Mario Advance
   Microsoft Train Simulator
   San Francisco Rush 2049
   WipeOut 64
   Dead or Alive
   Magic: The Gathering Arena
   Lair of the Clockwork God
   Snipperclips Plus: Cut It Out, Together!
   Tails Of Iron
   Hatsune Miku: Project Diva Future Tone - Colorful Tone
   Hatsune Miku: Project Diva Future Tone - Future Sound
   Dark Void Zero
   Etrian Odyssey IV: Legends of the Titan
   Black Mesa
   Rise of Nations: Rise 

KeyboardInterrupt: 

In [48]:
df = pd.DataFrame(products)

print(df)

print("Total products:", len(df))
print("Total listing pages:", len(visited_pages))

                             Product name    Price  Stock status  \
0    The Legend of Zelda: Ocarina of Time  91,99 €      in stock   
1                      Super Mario Galaxy  91,99 €  out of stock   
2                    Super Mario Galaxy 2  91,99 €      in stock   
3                           Metroid Prime  89,99 €  out of stock   
4                     Super Mario Odyssey  89,99 €      in stock   
..                                    ...      ...           ...   
411                        Rayman Advance  78,99 €  out of stock   
412                 Saints Row: The Third  77,99 €      in stock   
413                        Gran Turismo 5  78,99 €  out of stock   
414                              Tron 2.0  83,99 €      in stock   
415                        NieR: Automata  86,99 €  out of stock   

                                           Description  \
0    As a young boy, Link is tricked by Ganondorf, ...   
1    [Metacritic's 2007 Wii Game of the Year] The u...   
2    Supe

In [49]:
df = df.drop_duplicates(
    subset=["Detail-page URL"]
)

df = df.reset_index(drop=True)

print("Products after duplicate removal:", len(df))
print(df["Detail-page URL"].duplicated().sum())

Products after duplicate removal: 416
0


In [50]:
df.to_csv(filename, index=False)

print("CSV saved:", filename)

CSV saved: 23L_0570_versionA_static_products.csv


Question 1 – Scraping Methodology

How you identified the relevant elements/records on each page ?

answer:by inspecting each product card and finding what hierarchy or pattern each card has similar

How you navigated across pages

Answer:there is a forward button for pagination,

How your program decided that scraping was complete

Answer: when reaches the last page that forward button disables

Any challenges you ran into and how you resolved them.

Answer: while implementing the retry strategy for the failed requests.

How you verified that your scraper was actually collecting correct, complete data (spot checks,counts, etc.)?

Answer: ok so i checked each page and count of product on each page and then multiplied them and also dod spot checking that if that certain product is in that certain page. and at the start some of pages were scraped and some were not so had to work on them.

Question 1 – Validation Statistics

Total listing pages processed: 94

Total product URLs collected: 3000

Total records extracted with required fields: 3000

Duplicate or empty records removed/skipped: none

Failed requests retried and recovered: none

github link : https://github.com/atikaahussain/Web-Scraping-DataScience-Assignment